In [5]:
import pandas as pd

df1 = pd.read_csv('M18_건설업.csv')
df2 = pd.read_csv('M19_도매_소매업.csv')

df_all = pd.concat([df1, df2], ignore_index=True)

In [6]:
df_all.to_csv('M18,M19.csv', index=False)

In [ ]:
55f022874c5017a0ee5aa381c98c0f6ebc2a14ce

In [9]:
"""
DART OpenAPI를 활용한 재무데이터 결측치 보완 스크립트
대상: KS 11차 중분류 F41, F42, F43, G45, G46, G47 기업
보완 기준: NaN + 0

[ACCOUNT_MAP 설계 근거]
CSV 119개 재무 컬럼을 3가지로 분류:
  A) DART 직접 매핑 가능  → ACCOUNT_MAP에 포함 (아래 코드)
  B) DART에 없음 / 파생값  → 보완 불가, 주석으로 명시
  C) 폐지 항목            → 보완 대상 제외 (2007년 이전 경상이익 등)

[B 불가 항목 목록 - DART에서 제공 안 함]
  - CPA수정후 당기순이익          : 감사인 수정 후 값, DART 비제공
  - *당기순이익(요약)(백만원)      : 현금흐름표 내 중복 항목
  - 당기순이익(요약)(백만원).1    : pandas 중복컬럼 자동생성, 현흐표 내 항목
  - *감가상각비                   : 현금흐름 조정 항목, 별도 계정
  - *전기오류수정손익(x2)         : 전기 수정 항목
  - *법인세효과                   : 중단사업 법인세효과
  - *할인어음 또는 배서어음        : 우발부채성 주석항목
  - 렌탈자산 / 설비자산           : 업종 특수 세부항목
  - 이연자산 / 이연부채           : 구 회계기준 항목 (IFRS 전환 후 폐지)
  - 현금의 수입과 지출이 없는 거래 : 주석 공시 항목
  - 합병분할로 인한 증가(감소)     : 비현금 특수거래
  - 현금등가물                    : 현금흐름 조정 항목
  - 발행주식 액면가(원)            : 주식 기본정보, 재무제표 외 항목
"""

import pandas as pd
import requests
import time
import zipfile
import io
import xml.etree.ElementTree as ET
from typing import Optional

# ─── 설정 ──────────────────────────────────────────────────────────────────
DART_API_KEY = "55f022874c5017a0ee5aa381c98c0f6ebc2a14ce"
INPUT_FILE   = "M18,M19.csv"
OUTPUT_FILE  = "M18_M19_filled.csv"
TARGET_CODES = [41, 42, 43, 45, 46, 47]
MID_COL      = "통계청 한국표준산업분류 코드 11차(중분류)"
ANCHOR_COL   = "자산총계(요약)(백만원)"

# ─── ACCOUNT_MAP (전체 커버 버전) ───────────────────────────────────────────
# DART account_nm(좌) → CSV 컬럼명(우)
# 단위: DART는 원(₩) → 변환 시 /1,000,000 적용 (백만원 단위)
# 주당 지표는 원 단위 그대로 사용 (변환 없음 → convert_unit=False 플래그로 구분)

ACCOUNT_MAP = {
    # ── 재무상태표 ────────────────────────────────────────────────────────
    "자산총계":                        ("자산총계(요약)(백만원)",                                True),
    "유동자산":                        ("유동자산(요약)(백만원)",                                True),
    "당좌자산":                        ("당좌자산(요약)(백만원)",                                True),
    "현금및현금성자산":                 ("현금 및 현금성자산(요약)(백만원)",                      True),
    "단기투자자산":                     ("단기투자자산(요약)(백만원)",                            True),
    "단기예금":                        ("단기예금(요약)(백만원)",                                True),
    "단기매매증권":                     ("단기매매증권(요약)(백만원)",                            True),
    "단기대여금":                       ("단기대여금(요약)(백만원)",                              True),
    "기타단기투자자산":                 ("기타단기투자자산(요약)(백만원)",                        True),
    "매출채권":                        ("매출채권(요약)(백만원)",                                True),
    "기타당좌자산":                     ("기타당좌자산(요약)(백만원)",                            True),
    "재고자산":                        ("재고자산(요약)(백만원)",                                True),
    "상품":                            ("상품(요약)(백만원)",                                    True),
    "제품":                            ("제품(요약)(백만원)",                                    True),
    "반제품":                          ("반제품(요약)(백만원)",                                  True),
    "재공품":                          ("재공품(요약)(백만원)",                                  True),
    "원재료":                          ("원재료(요약)(백만원)",                                  True),
    "기타재고자산":                     ("기타재고자산(요약)(백만원)",                            True),
    "임대주택자산":                     ("임대주택자산(요약)(백만원)",                            True),
    "비유동자산":                       ("비유동자산(요약)(백만원)",                              True),
    "투자자산":                        ("투자자산(요약)(백만원)",                                True),
    "장기금융상품":                     ("*장기금융상품(요약)(백만원)",                           True),
    "장기투자증권":                     ("*투자유가증권(장기투자증권)(요약)(백만원)",              True),
    "유형자산":                        ("유형자산(요약)(백만원)",                                True),
    "토지":                            ("토지",                                                 True),
    "건물":                            ("건물",                                                 True),
    "기계장치":                        ("기계장치",                                              True),
    "차량운반구":                       ("차량운반구",                                           True),
    "건설중인자산":                     ("건설중인자산",                                         True),
    "무형자산":                        ("무형자산(요약)(백만원)",                                True),
    "기타비유동자산":                   ("기타비유동자산(요약)(백만원)",                          True),
    "이연자산":                        ("이연자산(요약)(백만원)",                                True),   # 구기준 항목, 있으면 채움
    "부채총계":                        ("부채총계(요약)(백만원)",                                True),
    "유동부채":                        ("유동부채(요약)(백만원)",                                True),
    "매입채무":                        ("매입채무(요약)(백만원)",                                True),
    "단기차입금":                       ("단기차입금(요약)(백만원)",                              True),
    "유동성장기부채":                   ("유동성장기부채(요약)(백만원)",                          True),
    "기타유동부채":                     ("기타유동부채(요약)(백만원)",                            True),
    "비유동부채":                       ("비유동부채(요약)(백만원)",                              True),
    "사채":                            ("사채(요약)(백만원)",                                    True),
    "장기차입금":                       ("장기차입금(요약)(백만원)",                              True),
    "제충당금":                        ("제충당금(요약)(백만원)",                                True),
    "기타비유동부채":                   ("기타비유동부채(요약)(백만원)",                          True),
    "이연부채":                        ("이연부채(요약)(백만원)",                                True),   # 구기준 항목
    "자본총계":                        ("자본총계(요약)(백만원)",                                True),
    "자본금":                          ("자본금(요약)(백만원)",                                  True),
    "자본잉여금":                       ("자본잉여금(요약)(백만원)",                              True),
    "자본조정":                        ("자본조정(요약)(백만원)",                                True),
    "기타포괄손익누계액":               ("기타포괄손익누계액(요약)(백만원)",                      True),
    "이익잉여금":                       ("이익잉여금(요약)(백만원)",                              True),
    "미처분이익잉여금":                 ("*미처분이익잉여금 및 차기이월미처분이익잉여금(요약)(백만원)", True),
    "부채와자본총계":                   ("부채와자본총계(요약)(백만원)",                          True),

    # ── 손익계산서 ────────────────────────────────────────────────────────
    "매출액":                          ("매출액(요약)(백만원)",                                  True),
    "매출원가":                        ("매출원가(요약)(백만원)",                                True),
    "매출총이익":                       ("매출총이익(요약)(백만원)",                              True),
    "판매비와관리비":                   ("판매비와 관리비(요약)(백만원)",                         True),
    "급여":                            ("급료",                                                 True),
    "퇴직급여":                        ("퇴직급여(요약)(백만원)",                                True),
    "복리후생비":                       ("복리후생비(요약)(백만원)",                              True),
    "세금과공과":                       ("세금과공과(요약)(백만원)",                              True),
    "임차료":                          ("임차료(요약)(백만원)",                                  True),
    "감가상각비":                       ("감가상각비",                                           True),
    "연구비":                          ("연구비",                                               True),
    "기타판매비와관리비":               ("기타 판매비와 관리비(요약)(백만원)",                    True),
    "영업이익":                        ("영업이익(요약)(백만원)",                                True),
    "영업외수익":                       ("영업외수익(요약)(백만원)",                              True),
    "이자수익":                        ("이자수익(요약)(백만원)",                                True),
    "배당금수익":                       ("배당금수익(요약)(백만원)",                              True),
    "외환차익":                        ("외환차익(요약)(백만원)",                                True),
    "외화환산이익":                     ("외화환산이익(요약)(백만원)",                            True),
    "지분법이익":                       ("지분법이익(요약)(백만원)",                              True),
    "기타영업외수익":                   ("기타영업외수익(요약)(백만원)",                          True),
    "영업외비용":                       ("영업외비용(요약)(백만원)",                              True),
    "이자비용":                        ("이자비용(요약)(백만원)",                                True),
    "외환차손":                        ("외환차손(요약)(백만원)",                                True),
    "외화환산손실":                     ("외화환산손실(요약)(백만원)",                            True),
    "지분법손실":                       ("지분법손실(요약)(백만원)",                              True),
    "기타영업외비용":                   ("기타영업외비용(요약)(백만원)",                          True),
    "법인세비용차감전계속사업손익":      ("법인세비용차감전(계속사업)손익(요약)(백만원)",           True),
    "계속사업손익법인세비용":            ("(계속사업손익)법인세비용(요약)(백만원)",                True),
    "계속사업이익":                     ("계속사업이익(요약)(백만원)",                            True),
    "중단사업이익":                     ("중단사업이익(요약)(백만원)",                            True),
    "당기순이익":                       ("당기순이익(요약)(백만원)",                              True),
    # 주당 지표 - 원 단위 그대로 (convert_unit=False)
    "주당계속사업이익":                 ("주당계속사업이익(요약)(원)",                            False),
    "주당순이익":                       ("주당순이익(요약)(원)",                                  False),
    "희석화주당계속사업이익":           ("희석화주당계속사업이익(요약)(원)",                      False),
    "희석화주당순이익":                 ("희석화주당순이익(요약)(원)",                            False),

    # ── 현금흐름표 ────────────────────────────────────────────────────────
    "영업활동으로인한현금흐름":          ("영업활동으로 인한 현금흐름(요약)(백만원)",              True),
    "현금의유출이없는비용등가산":        ("현금의 유출이 없는 비용 등 가산(요약)(백만원)",         True),
    "현금의유입이없는수익등차감":        ("현금의 유입이없는 수익등의 차감(요약)(백만원)",         True),
    "영업활동자산및부채변동":           ("영업활동으로 인한 자산 및 부채의변동(요약)(백만원)",     True),
    "영업활동자산감소증가":             ("영업활동으로 인한 자산의 감소(증가)(요약)(백만원)",      True),
    "영업활동부채증가감소":             ("영업활동으로 인한 부채의 증가(감소)(요약)(백만원)",      True),
    "투자활동으로인한현금흐름":          ("투자활동으로 인한 현금흐름(요약)(백만원)",              True),
    "투자활동현금유입액":               ("투자활동으로 인한 현금유입액(요약)(백만원)",             True),
    "투자활동현금유출액":               ("투자활동으로 인한 현금유출액(요약)(백만원)",             True),
    "재무활동으로인한현금흐름":          ("재무활동으로 인한 현금흐름(요약)(백만원)",              True),
    "재무활동현금유입액":               ("재무활동으로 인한 현금유입액(요약)(백만원)",             True),
    "재무활동현금유출액":               ("재무활동으로 인한 현금유출액(요약)(백만원)",             True),
    "환율변동으로인한차이조정":          ("환율변동으로 인한 차이조정(요약)(백만원)",              True),
    "현금의증가감소":                   ("현금의 증가(감소)(요약)(백만원)",                       True),
    "기초현금":                        ("기초현금(요약)(백만원)",                                True),
    "기말현금":                        ("기말현금(요약)(백만원)",                                True),
}

# DART에서 제공하지 않아 보완 불가한 컬럼 (참고용)
UNMAPPABLE_COLS = [
    "CPA수정후 당기순이익(요약)(백만원)",           # 감사인 수정값, DART 비제공
    "당기순이익(요약)(백만원).1",                  # pandas 중복컬럼(현금흐름표 내 항목)
    "*당기순이익(요약)(백만원)",                   # 현금흐름표 내 별도 표시
    "*감가상각비",                                # 현금흐름 조정 내 별도 항목
    "*전기오류수정손익(요약)(백만원)",              # 전기 수정 항목
    "*전기오류수정손익(요약)(백만원).1",
    "*법인세효과(요약)(백만원)",                   # 중단사업 법인세효과
    "*할인어음 또는 배서어음(요약)(백만원)",         # 우발부채성 주석항목
    "렌탈자산",                                   # 업종 특수 세부항목
    "설비자산",
    "현금의 수입과 지출이 없는 거래(요약)(백만원)", # 주석 공시 항목
    "합병분할(영업양수도)으로 인한 증가(감소)(요약)(백만원)",  # 비현금 특수거래
    "현금등가물(요약)(백만원)",                    # 현금흐름 조정 항목
    "발행주식 액면가(원)",                         # 주식기본정보, 재무제표 외
    "*주당경상이익(2007년 이전 발생)(요약)(원)",    # 2008년 이후 폐지
    "*희석화주당경상이익(2007년 이전 발생)(요약)(원)",
    "*미처분이익잉여금 및 차기이월미처분이익잉여금(요약)(백만원)",  # 위 MAP에 추가됨
]


# ─── 보완 판단 함수 ─────────────────────────────────────────────────────────
def needs_fill(value) -> bool:
    if pd.isnull(value):
        return True
    if value == 0:
        return True
    return False


# ─── Step 1. 데이터 로드 및 결측 대상 추출 ──────────────────────────────────
def load_and_identify_missing(input_file: str) -> tuple:
    df = pd.read_csv(input_file)
    target_df = df[df[MID_COL].isin(TARGET_CODES)].copy()
    missing_mask = target_df[ANCHOR_COL].isnull() | (target_df[ANCHOR_COL] == 0)
    missing_rows = target_df[missing_mask].copy()

    nan_cnt  = target_df[ANCHOR_COL].isnull().sum()
    zero_cnt = (target_df[ANCHOR_COL] == 0).sum()
    print(f"[INFO] 대상 행: {len(target_df):,} | NaN: {nan_cnt}행 | Zero: {zero_cnt}행 | 보완 대상: {len(missing_rows)}행")
    print(f"[INFO] ACCOUNT_MAP 커버 컬럼: {len(ACCOUNT_MAP)}개 / 전체 재무 컬럼: 119개")
    print(f"[INFO] DART 미제공으로 보완 불가: {len(UNMAPPABLE_COLS)}개 컬럼")
    return df, missing_rows


# ─── Step 2. 종목코드 → DART corp_code 매핑 ─────────────────────────────────
def build_corp_code_map(stock_codes: list, api_key: str) -> dict:
    url = "https://opendart.fss.or.kr/api/corpCode.xml"
    resp = requests.get(url, params={"crtfc_key": api_key}, timeout=30)
    resp.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        with z.open("CORPCODE.xml") as f:
            tree = ET.parse(f)

    corp_map = {}
    for item in tree.getroot().findall("list"):
        stock_cd = item.findtext("stock_code", "").strip()
        corp_cd  = item.findtext("corp_code", "").strip()
        if stock_cd and corp_cd:
            try:
                corp_map[int(stock_cd)] = corp_cd
            except ValueError:
                pass

    found     = {sc: corp_map[sc] for sc in stock_codes if sc in corp_map}
    not_found = [sc for sc in stock_codes if sc not in corp_map]
    print(f"[INFO] corp_code 매핑 성공: {len(found)} | 실패: {len(not_found)}")
    if not_found:
        print(f"  ※ 매핑 실패 종목코드: {not_found}")
    return found


# ─── Step 3. DART 재무제표 API 호출 ─────────────────────────────────────────
def fetch_dart_financials(corp_code: str, year: int, api_key: str,
                          reprt_code: str = "11011") -> Optional[pd.DataFrame]:
    url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
    for fs_div in ["OFS", "CFS"]:
        params = {
            "crtfc_key":  api_key,
            "corp_code":  corp_code,
            "bsns_year":  str(year),
            "reprt_code": reprt_code,
            "fs_div":     fs_div,
        }
        try:
            resp = requests.get(url, params=params, timeout=15)
            resp.raise_for_status()
            data = resp.json()
            if data.get("status") == "000" and data.get("list"):
                df = pd.DataFrame(data["list"])
                df["fs_div_used"] = fs_div
                return df
        except Exception as e:
            print(f"  [WARN] {corp_code}/{year}/{fs_div}: {e}")
        time.sleep(0.2)
    return None


# ─── Step 4. DART 응답 → 컬럼값 추출 ────────────────────────────────────────
def extract_values_from_dart(dart_df: pd.DataFrame) -> dict:
    result = {}
    for dart_nm, (csv_col, convert) in ACCOUNT_MAP.items():
        match = dart_df[dart_df["account_nm"].str.strip() == dart_nm]
        if not match.empty:
            raw = match.iloc[0]["thstrm_amount"]
            try:
                val = float(str(raw).replace(",", "").replace(" ", ""))
                result[csv_col] = round(val / 1_000_000, 2) if convert else val
            except (ValueError, TypeError):
                result[csv_col] = None
    return result


# ─── Step 5. 결측 보완 메인 루프 ─────────────────────────────────────────────
def fill_missing_values(df: pd.DataFrame, missing_rows: pd.DataFrame,
                        corp_code_map: dict, api_key: str) -> pd.DataFrame:
    filled_log = []

    for idx, row in missing_rows.iterrows():
        stock_code = int(row["거래소코드"])
        company    = row["회사명"]
        year       = int(str(row["회계년도"])[:4])
        anchor_val = row[ANCHOR_COL]
        reason     = "NaN" if pd.isnull(anchor_val) else "Zero"

        corp_code = corp_code_map.get(stock_code)
        if not corp_code:
            print(f"  [SKIP] {company}({stock_code}) {year} - corp_code 없음")
            filled_log.append({"회사명": company, "연도": year, "기존값": reason, "status": "corp_code_없음"})
            continue

        print(f"  [FETCH] {company}({stock_code}) {year}년 [{reason}] ...", end=" ")
        dart_df = fetch_dart_financials(corp_code, year, api_key)

        if dart_df is None:
            print("→ DART 데이터 없음")
            filled_log.append({"회사명": company, "연도": year, "기존값": reason, "status": "API_데이터없음"})
            continue

        values = extract_values_from_dart(dart_df)
        filled_count = 0
        for col, val in values.items():
            if needs_fill(df.at[idx, col]) and val is not None:
                df.at[idx, col] = val
                filled_count += 1

        print(f"→ {filled_count}개 항목 보완")
        filled_log.append({"회사명": company, "연도": year, "기존값": reason,
                           "status": "성공", "보완항목수": filled_count})
        time.sleep(0.3)

    print("\n[완료 요약]")
    print(pd.DataFrame(filled_log).to_string(index=False))
    return df


# ─── Step 6. 검증 ────────────────────────────────────────────────────────────
def validate_fill(df_before: pd.DataFrame, df_after: pd.DataFrame) -> None:
    fin_cols = [c for c in df_before.columns if "백만원" in c or c in ["토지","건물","기계장치","차량운반구","건설중인자산","급료","감가상각비","연구비"]]
    before_nan  = df_before[fin_cols].isnull().sum().sum()
    after_nan   = df_after[fin_cols].isnull().sum().sum()
    before_zero = (df_before[fin_cols] == 0).sum().sum()
    after_zero  = (df_after[fin_cols] == 0).sum().sum()
    print(f"\n[검증] NaN:  {before_nan:,} → {after_nan:,}  (감소: {before_nan  - after_nan:,})")
    print(f"[검증] Zero: {before_zero:,} → {after_zero:,}  (감소: {before_zero - after_zero:,})")


# ─── Main ────────────────────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("DART 결측치(NaN + Zero) 보완 시작")
    print("=" * 60)

    df, missing_rows = load_and_identify_missing(INPUT_FILE)
    df_backup = df.copy()

    if missing_rows.empty:
        print("[INFO] 보완 대상 없음. 종료.")
        return

    missing_stock_codes = missing_rows["거래소코드"].unique().tolist()
    print(f"\n결측 기업 수: {len(missing_stock_codes)}개사\n")

    print("[Step 1] corp_code 매핑 중...")
    corp_code_map = build_corp_code_map(missing_stock_codes, DART_API_KEY)

    print("\n[Step 2] DART 재무데이터 조회 및 보완 중...")
    df_filled = fill_missing_values(df, missing_rows, corp_code_map, DART_API_KEY)

    validate_fill(df_backup, df_filled)
    df_filled.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(f"\n[저장 완료] {OUTPUT_FILE}")
    print("=" * 60)


if __name__ == "__main__":
    main()

DART 결측치(NaN + Zero) 보완 시작
[INFO] 대상 행: 91,069 | NaN: 50행 | Zero: 6행 | 보완 대상: 56행
[INFO] ACCOUNT_MAP 커버 컬럼: 103개 / 전체 재무 컬럼: 119개
[INFO] DART 미제공으로 보완 불가: 17개 컬럼

결측 기업 수: 19개사

[Step 1] corp_code 매핑 중...
[INFO] corp_code 매핑 성공: 10 | 실패: 9
  ※ 매핑 실패 종목코드: [40249, 74459, 83724, 83720, 10079, 85541, 83743, 95376, 60702]

[Step 2] DART 재무데이터 조회 및 보완 중...
  [FETCH] (주)베노티앤알(206400) 2014년 [NaN] ... → DART 데이터 없음
  [SKIP] 엘시종합건설주식회사(40249) 2014 - corp_code 없음
  [SKIP] 주식회사낙원종합건설(74459) 2016 - corp_code 없음
  [SKIP] 주식회사다원에이앤씨(83724) 2016 - corp_code 없음
  [FETCH] (주)대산에프앤비(65150) 2018년 [Zero] ... → 23개 항목 보완
  [FETCH] (주)더블유에스아이(299170) 2018년 [NaN] ... → DART 데이터 없음
  [FETCH] (주)더블유에스아이(299170) 2019년 [NaN] ... → DART 데이터 없음
  [FETCH] (주)디와이디(219550) 2015년 [NaN] ... → DART 데이터 없음
  [FETCH] (주)디와이디(219550) 2016년 [NaN] ... → 17개 항목 보완
  [FETCH] (주)윙스풋(335870) 2019년 [NaN] ... → DART 데이터 없음
  [FETCH] (주)윙스풋(335870) 2020년 [NaN] ... → DART 데이터 없음
  [FETCH] (주)윙스풋(335870) 2021년 [NaN] ... → DART 데이터 없음